In [ ]:
%%configure -f
{
  "defaultLakehouse": {
    "name": {
      "parameterName": "lakehouse_name",
      "defaultValue": "YourLakehouse"
    }
  }
}

# OLAF Runner — pipeline entry, one activity = one mode

Point a Fabric / ADF **Notebook activity** at this notebook and pass `mode` (plus any overrides)
as base parameters. The `%%configure` cell above binds the run's default lakehouse **by name** —
`lakehouse_name` is the same base parameter the parameters cell below receives, so one pipeline
parameter drives both the session binding and `setup`'s assertion — then a single
`notebookutils.notebook.run` dispatches `olaf`, which must live in the **same workspace**, and
this notebook exits with olaf's result envelope for the pipeline to branch on. Interactive runs
can attach a lakehouse in the portal instead; `defaultValue` above is what running the cell binds.
Fail-visible by design: if no binding lands, olaf blocks with `no lakehouse attached` — it never
writes anywhere unintended.

In [ ]:
# PARAMETERS — tagged "parameters": a pipeline's base parameters land here (String-typed;
# olaf coerces the bool-ish ones itself, and REJECTS ambiguous values with a blocked envelope).
# This surface MIRRORS olaf's own parameters cell — full per-parameter semantics live there —
# plus timeout_seconds, which belongs to this notebook's child run, not to olaf.
# fmt: off
mode                = ""                                  # [required] setup | generate | validate | plan | apply | rollback | show | trace
rebuild             = False                               # generate: rebuild even if config unchanged · setup: recreate drifted tables
keep_unmanaged      = False                               # apply: False = full REPLACE · True = incremental upsert
if_match            = True                                # apply/rollback: conditional bulk PUT (False = the documented escape hatch)
control_data_isolation_attestation = ""                    # required for each sensitive write run; external evidence reference, not a secret
tenant_id           = ""                                  # "" = auto-resolve from the runtime context
lakehouse_name      = "YourLakehouse"                     # setup's assertion — and the %%configure binding above
config_table        = "olaf.onelake_security_config"
mapping_table       = "olaf.onelake_security_mapping"
log_table           = "olaf.onelake_security_log"
member_table        = "olaf.onelake_security_member"
mapping_history_dir = "Files/security/mapping-history"
role_backup_dir     = "Files/security/role-backups"
verbosity           = "info"                              # silent | quiet | info | detail | verbose
env                 = ""                                  # "" = unset (olaf's own default) · dev | qa | prod — written to every log row
batch_id            = ""                                  # "" = new uuid · pass the pipeline run id to link plan → apply
by                  = "table"                             # show: table | role | member
subject             = ""                                  # show: the pivot subject (globs allowed)
rollback_to_version = ""                                  # rollback: "" = previous config version · N = that Delta version
rollback_reason     = ""                                  # rollback: required — stamped into the audit log
timeout_seconds     = 3600                                # ceiling for the child olaf run (runner-only, not passed to olaf)
# fmt: on

In [ ]:
# ── dispatch — one notebook.run, the envelope handed straight to the pipeline ─────────────────
# The args dict REPLACES olaf's parameters cell wholesale, so EVERY parameter above is passed
# explicitly — an omitted key would silently fall back to olaf's own default, not this cell's.
# olaf RAISES on a blocked/error outcome (a compact JSON payload), and that raise is left to
# propagate through notebook.run on purpose: it is what fails this activity, so a pipeline
# branches on the native Failure arrow. On success/skipped, olaf exits with the full result
# envelope — hand it up unchanged as this notebook's own exit value.
import notebookutils

result = notebookutils.notebook.run(
    "olaf",
    timeout_seconds,
    {
        "mode": mode,
        "rebuild": rebuild,
        "keep_unmanaged": keep_unmanaged,
        "if_match": if_match,
        "control_data_isolation_attestation": control_data_isolation_attestation,
        "tenant_id": tenant_id,
        "lakehouse_name": lakehouse_name,
        "config_table": config_table,
        "mapping_table": mapping_table,
        "log_table": log_table,
        "member_table": member_table,
        "mapping_history_dir": mapping_history_dir,
        "role_backup_dir": role_backup_dir,
        "verbosity": verbosity,
        "env": env,
        "batch_id": batch_id,
        "by": by,
        "subject": subject,
        "rollback_to_version": rollback_to_version,
        "rollback_reason": rollback_reason,
    },
)
notebookutils.notebook.exit(result)
